In [14]:
"""
Решение контрольной работы №4
Поиск аномальных респондентов в активности поисковых запросов SoS

Алгоритм: Robust Z-Score на основе медианы и MAD
Выполнила: Москалева Екатерина Андреевна
Группа: ШЦТ-111
"""


'\nРешение контрольной работы №4\nПоиск аномальных респондентов в активности поисковых запросов SoS\n\nАлгоритм: Robust Z-Score на основе медианы и MAD\nВыполнила: Москалева Екатерина Андреевна\nГруппа: ШЦТ-111\n'

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import median_abs_deviation
import warnings
warnings.filterwarnings('ignore')

In [16]:
Z_THRESHOLD = 8.0
MIN_OBSERVATIONS = 10

In [17]:
print("\n1. Загрузка данных")
data_path = Path(r"C:\Users\Kate\OneDrive\Desktop\4 полусем\data_train")

all_dfs = []
for month_dir in data_path.glob("month=*"):
    if month_dir.is_dir():
        for parquet_file in month_dir.glob("*.parquet"):
            df_temp = pd.read_parquet(parquet_file)
            all_dfs.append(df_temp)

df = pd.concat(all_dfs, ignore_index=True)
print(f"   Всего: {len(df):,} строк")


1. Загрузка данных
   Всего: 280,458 строк


In [18]:
print("\n2. Предобработка")
if 'CategoryNameDelivery' in df.columns:
    df = df.rename(columns={'CategoryNameDelivery': 'CategoryDelivery'})

df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
mask = (df['BrandinDelivery'] == 1) & (df['CategoryDelivery'].notna())
df_filtered = df[mask].copy()
print(f"   Строк после фильтрации: {len(df_filtered):,}")


2. Предобработка
   Строк после фильтрации: 255,748


In [19]:
print("\n3. Агрегация daily_ots")
grouped = df_filtered.groupby(
    ['SubjectID', 'researchdate', 'CategoryDelivery', 'BrandID', 'Brand']
).size().reset_index(name='query_count')

weights = df_filtered[['SubjectID', 'researchdate', 'Weight']].drop_duplicates()
daily_ots_df = grouped.merge(weights, on=['SubjectID', 'researchdate'], how='left')
daily_ots_df['daily_ots'] = daily_ots_df['Weight'] * daily_ots_df['query_count']
print(f"   Уникальных: {len(daily_ots_df):,}")


3. Агрегация daily_ots
   Уникальных: 175,833


In [20]:
print(f"\n4. Поиск аномалий (Z-Score > {Z_THRESHOLD})")

anomaly_records = []
stats_by_brand = []

for (category, brand_id), group in daily_ots_df.groupby(['CategoryDelivery', 'BrandID']):
    if len(group) < MIN_OBSERVATIONS:
        continue
    
    ots_values = group['daily_ots'].values
    median_val = np.median(ots_values)
    mad_val = median_abs_deviation(ots_values)
    
    if mad_val < 1e-9:
        continue
    
    brand_anomalies = 0
    
    for _, row in group.iterrows():
        z_score = (row['daily_ots'] - median_val) / mad_val
        
        if z_score > Z_THRESHOLD:
            brand_anomalies += 1
            anomaly_records.append({
                'SubjectID': row['SubjectID'],
                'researchdate': row['researchdate'],
                'BrandID': row['BrandID'],
                'Brand': row['Brand'],
                'CategoryDelivery': row['CategoryDelivery'],
                'daily_ots': row['daily_ots'],
                'score': z_score,
                'threshold': Z_THRESHOLD,
                'median_ots': median_val,
                'mad_ots': mad_val,
                'reason': f'Robust Z-score = {z_score:.2f} > {Z_THRESHOLD} (median={median_val:.0f}, MAD={mad_val:.0f})'
            })
    
    stats_by_brand.append({
        'CategoryDelivery': category,
        'BrandID': brand_id,
        'Brand': group['Brand'].iloc[0],
        'total_observations': len(group),
        'anomalies_count': brand_anomalies,
        'median_ots': median_val,
        'mad_ots': mad_val,
        'max_ots': ots_values.max(),
        'max_z_score': (ots_values.max() - median_val) / mad_val if mad_val > 0 else 0
    })

anomaly_reasons = pd.DataFrame(anomaly_records)
stats_df = pd.DataFrame(stats_by_brand).sort_values('anomalies_count', ascending=False)

print(f"   Найдено аномалий: {len(anomaly_reasons)}")


4. Поиск аномалий (Z-Score > 8.0)
   Найдено аномалий: 8005


In [21]:
print("\n5. Формирование списка респондентов на удаление")
users_to_remove = anomaly_reasons[['SubjectID', 'researchdate']].drop_duplicates()
print(f"   Удаляется пар (респондент-день): {len(users_to_remove)}")


5. Формирование списка респондентов на удаление
   Удаляется пар (респондент-день): 6764


In [22]:
print("\n6. Сохранение результатов")
Path('output/plots').mkdir(parents=True, exist_ok=True)

users_to_remove.to_csv('output/anomalies.csv', index=False)
anomaly_reasons.to_csv('output/anomaly_reasons.csv', index=False)
stats_df.to_csv('output/brand_statistics.csv', index=False)

print(f"   anomalies.csv: {len(users_to_remove)} записей")
print(f"   anomaly_reasons.csv: {len(anomaly_reasons)} записей")


6. Сохранение результатов
   anomalies.csv: 6764 записей
   anomaly_reasons.csv: 8005 записей


In [23]:
plt.figure(figsize=(12, 5))

ots_before = daily_ots_df.groupby('researchdate')['daily_ots'].sum()
users_set = set(zip(users_to_remove['SubjectID'], users_to_remove['researchdate']))
daily_ots_df['is_removed'] = daily_ots_df.apply(
    lambda r: (r['SubjectID'], r['researchdate']) in users_set, axis=1
)
ots_after = daily_ots_df[~daily_ots_df['is_removed']].groupby('researchdate')['daily_ots'].sum()

all_dates = sorted(set(ots_before.index) | set(ots_after.index))
before_vals = [ots_before.get(d, 0) for d in all_dates]
after_vals = [ots_after.get(d, 0) for d in all_dates]

plt.plot(all_dates, before_vals, 'b-o', label='До удаления', linewidth=2, markersize=6)
plt.plot(all_dates, after_vals, 'r-s', label='После удаления', linewidth=2, markersize=6)
plt.xlabel('Дата')
plt.ylabel('Суммарный OTS')
plt.title(f'Изменение общего OTS (порог Z-Score > {Z_THRESHOLD})')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('output/plots/total_ots_before_after.png', dpi=150)
plt.close()

In [24]:
plt.figure(figsize=(12, 6))

category_ots_before = daily_ots_df.groupby('CategoryDelivery')['daily_ots'].sum()
category_ots_after = daily_ots_df[~daily_ots_df['is_removed']].groupby('CategoryDelivery')['daily_ots'].sum()

category_change = {}
for cat in category_ots_before.index:
    before = category_ots_before.get(cat, 0)
    after = category_ots_after.get(cat, 0)
    if before > 0:
        change_pct = (1 - after / before) * 100
        category_change[cat] = change_pct

category_change = pd.Series(category_change).sort_values(ascending=False).head(15)
colors = ['green' if x < 5 else 'orange' if x < 20 else 'red' for x in category_change.values]
plt.barh(range(len(category_change)), category_change.values, color=colors)
plt.yticks(range(len(category_change)), category_change.index)
plt.xlabel('Снижение OTS (%)')
plt.title(f'Изменение OTS по категориям (порог > {Z_THRESHOLD})')
plt.tight_layout()
plt.savefig('output/plots/category_ots_change.png', dpi=150)
plt.close()

In [25]:
plt.figure(figsize=(12, 5))
daily_count = users_to_remove.groupby('researchdate').size()
if len(daily_count) > 0:
    plt.bar(daily_count.index.astype(str), daily_count.values, color='darkred', alpha=0.7)
else:
    plt.text(0.5, 0.5, 'Нет аномальных респондентов', ha='center', va='center', fontsize=14)
plt.xlabel('Дата')
plt.ylabel('Количество аномальных респондентов')
plt.title(f'Аномальные респонденты по дням (порог > {Z_THRESHOLD})')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('output/plots/daily_anomaly_count.png', dpi=150)
plt.close()

In [27]:
# Поисковые запросы аномального респондента

if len(anomaly_reasons) > 0:
    top_user = anomaly_reasons.groupby('SubjectID').size().sort_values(ascending=False).index[0]
    top_date = anomaly_reasons[anomaly_reasons['SubjectID'] == top_user]['researchdate'].iloc[0]
    
    print(f"\nРеспондент: {top_user}")
    print(f"Дата: {top_date}")
    
    user_queries = df[(df['SubjectID'] == top_user) & (df['researchdate'] == top_date) & (df['BrandinDelivery'] == 1)]
    print(f"\nПоисковые запросы (всего {len(user_queries)}):")
    for _, row in user_queries.head(20).iterrows():
        print(f"   - {row['QueryText'][:80]} (бренд: {row['Brand']})")
else:
    print("Аномалий не найдено")


Респондент: 7576643904375722625
Дата: 2025-10-19

Поисковые запросы (всего 19):
   - anker soundcore liberty 4 nc (бренд: soundcore)
   - old spice (бренд: old spice)
   - куртка демисезонная reserved мужская (бренд: reserved)
   - свитшот gucci (бренд: gucci)
   - худи tommy hilfiger (бренд: tommy hilfiger)
   - anker soundcore liberty 4 (бренд: anker)
   - anker soundcore liberty 4 (бренд: anker)
   - oneolus buds nord 3 vs realme buds t 310 (бренд: oneplus)
   - oneplus buds 3 (бренд: oneplus)
   - oneplus buds 4 (бренд: oneplus)
   - oneplus nord buds 3 (бренд: oneplus)
   - adidas puffylette (бренд: adidas)
   - yeezy (бренд: adidas)
   - crocs мужские (бренд: crocs)
   - crocs мужские (бренд: crocs)
   - realme buds air 7 (бренд: realme)
   - realme buds air 7 pro (бренд: realme)
   - realme buds t 310 (бренд: realme)
   - realme buds air 6 pro (бренд: realme)


In [28]:
print(f"Всего строк: {len(df):,}")
print(f"Валидных строк: {len(df_filtered):,}")
print(f"Уникальных (респондент-день-бренд): {len(daily_ots_df):,}")
print(f"\nНайдено аномалий: {len(anomaly_reasons)}")
print(f"Удаляется пар (респондент-день): {len(users_to_remove)}")
print(f"Общее снижение OTS: {(1 - ots_after.sum() / ots_before.sum()) * 100:.2f}%")

print("\nТоп-5 брендов по количеству аномалий:")
print(stats_df.head(5)[['Brand', 'CategoryDelivery', 'anomalies_count']].to_string())

Всего строк: 280,458
Валидных строк: 255,748
Уникальных (респондент-день-бренд): 175,833

Найдено аномалий: 8005
Удаляется пар (респондент-день): 6764
Общее снижение OTS: 30.59%

Топ-5 брендов по количеству аномалий:
               Brand CategoryDelivery  anomalies_count
1097    apple iphone        Смартфоны              494
1155  samsung galaxy        Смартфоны              434
1112           honor        Смартфоны              245
1137           redmi        Смартфоны              230
1135          realme        Смартфоны              208
